# Lab01 - Intro to CUDA

In [1]:
import numpy as np
import numba
import time
from numba import cuda
from wurlitzer import sys_pipes

cuda.detect()

Found 1 CUDA devices
id 0    b'NVIDIA GeForce RTX 2060'                              [SUPPORTED]
                      Compute Capability: 7.5
                           PCI Device ID: 0
                              PCI Bus ID: 1
                                    UUID: GPU-006291b6-94bd-bac8-588d-4a97f26284e1
                                Watchdog: Enabled
                            Compute Mode: WDDM
             FP32/FP64 Performance Ratio: 32
Summary:
	1/1 devices are supported


True

## Ex.1) Parallel Vector Sum

In [2]:
np.random.seed(42)
SIZE = 1024 * 1024

a_host = np.random.randint(0, 100, size=SIZE, dtype=np.int16)
b_host = np.random.randint(0, 100, size=SIZE, dtype=np.int16)

print(f"a: {a_host}")
print(f"b: {b_host}")

a: [97 51 92 ... 77 86 31]
b: [ 2 59 12 ... 42 10 73]


In [3]:
# Sequential Version

c_host_solution = np.zeros(SIZE, dtype=np.int16)

def sum(a, b, c):
    for i in range(SIZE):
        c[i] = a[i] + b[i]

start = time.time()
sum(a_host, b_host, c_host_solution)
end = time.time()

print(f"a+b: {c_host_solution}")
print(f"time: {end - start}s")

a+b: [ 99 110 104 ... 119  96 104]
time: 0.10843825340270996s


In [14]:
# Parallelized Version

BLOCK_SIZE = 32
GRID_SIZE = (SIZE + BLOCK_SIZE - 1) // BLOCK_SIZE

print(f"BLOCK_SIZE={BLOCK_SIZE}")
print(f"GRID_SIZE={GRID_SIZE}")

c_host = np.zeros(SIZE, dtype=np.int16)

a_device = cuda.to_device(a_host)
b_device = cuda.to_device(b_host)
c_device = cuda.device_array_like(c_host)

@cuda.jit
def parallel_sum(a, b, c):
    i = cuda.grid(1)
    if i < len(c):
        c[i] = a[i] + b[i]
    
start = time.time()
with sys_pipes():
    parallel_sum[GRID_SIZE, BLOCK_SIZE](a_device, b_device, c_device)
    cuda.synchronize()
end = time.time()

c_host = c_device.copy_to_host()

print(f"a+b: {c_host}")
print(f"time: {end - start}s")
assert all([x == y for x,y in zip(c_host, c_host_solution)])

BLOCK_SIZE=32
GRID_SIZE=32768
a+b: [ 99 110 104 ... 119  96 104]
time: 0.05892634391784668s


## Ex. 2) Fibonacci

🔹 **Task Description**

-   You must write a CUDA kernel that:
    1.  Computes s (the thread index) using thread and block indices
    2.  Checks whether s is a Fibonacci number
    3.  Write 1 only if s is Fibonacci


N is fibonacci IFF (5 N^2 + 4) or (5N^2 – 4) are perfect squares

In [ ]:
SIZE = 1024

out_host = np.zeros(SIZE)

In [20]:
import math 

BLOCK_SIZE = 32
GRID_SIZE = (SIZE + BLOCK_SIZE - 1) // BLOCK_SIZE

print(f"BLOCK_SIZE={BLOCK_SIZE}")
print(f"GRID_SIZE={GRID_SIZE}")

out_device = cuda.device_array_like(out_host)

@cuda.jit(device=True, inline=True)
def is_perfect_square(n):
    return math.floor(math.sqrt(n)) ** 2 == n 

@cuda.jit(device=True, inline=True)
def is_fibonacci(n):
    return is_perfect_square(5 * n * n + 4) or is_perfect_square(5 * n * n - 4)

@cuda.jit
def find_fibonacci_cuda(out):
    idx = cuda.grid(1)
    if idx < len(out):
        if is_fibonacci(idx):
            out[idx] = 1


with sys_pipes():
    find_fibonacci_cuda[GRID_SIZE, BLOCK_SIZE](out_device)
    cuda.synchronize()
    
out_host = out_device.copy_to_host()
for idx, i in enumerate(out_host):
    if i != 0:
        print(f"{idx} is a fibonacci number")


BLOCK_SIZE=32
GRID_SIZE=32


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 32 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


0 is a fibonacci number
1 is a fibonacci number
2 is a fibonacci number
3 is a fibonacci number
5 is a fibonacci number
8 is a fibonacci number
13 is a fibonacci number
21 is a fibonacci number
34 is a fibonacci number
55 is a fibonacci number
89 is a fibonacci number
144 is a fibonacci number
233 is a fibonacci number
377 is a fibonacci number
610 is a fibonacci number
987 is a fibonacci number
